# Notebook 05 — Random Forest Training
Trains the RF baseline on flat (non-windowed) features. No GPU needed.

In [ ]:
import subprocess, os
from pathlib import Path

REPO_URL  = 'https://github.com/calvinkatoroy/tugas-akhir-ai-kel-08.git'
REPO_NAME = 'tugas-akhir-ai-kel-08'

cwd = Path.cwd()

if (cwd / '../src').resolve().exists():
    result = subprocess.run(['git', '-C', str((cwd / '..').resolve()), 'pull'],
                            capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')

elif (cwd / 'src').exists():
    result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')
    os.chdir('notebooks')

else:
    repo_path = cwd / REPO_NAME
    if not repo_path.exists():
        print(f'Cloning {REPO_URL} ...')
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    else:
        print('Repo found, pulling latest...')
        subprocess.run(['git', '-C', str(repo_path), 'pull'], capture_output=True)
    os.chdir(repo_path / 'notebooks')

print(f'Working dir: {Path.cwd()}')

In [ ]:
import sys
sys.path.insert(0, '..')

import random
import numpy as np
import yaml
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
from google.colab import drive
from pathlib import Path

IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/tugas-akhir-ai")
    SPLITS_DIR = DRIVE_ROOT / "splits"
else:
    import yaml
    with open("../config.yaml") as f:
        _cfg = yaml.safe_load(f)
    SPLITS_DIR = Path("..") / _cfg["data"]["splits_path"]
    DRIVE_ROOT = SPLITS_DIR.parent
print(f"SPLITS_DIR: {SPLITS_DIR}")

In [ ]:
with open("../config.yaml") as f:
    cfg = yaml.safe_load(f)

# RF uses flat (non-windowed) splits
X_train = np.load(SPLITS_DIR / "X_train.npy")
y_train = np.load(SPLITS_DIR / "y_train.npy")
X_val   = np.load(SPLITS_DIR / "X_val.npy")
y_val   = np.load(SPLITS_DIR / "y_val.npy")
X_test  = np.load(SPLITS_DIR / "X_test.npy")
y_test  = np.load(SPLITS_DIR / "y_test.npy")

print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

## Baseline run (config defaults)

In [ ]:
from src.models.rf_model import build_rf, train_rf, save_rf
from src.utils import log_experiment
from sklearn.metrics import f1_score

rf = build_rf(cfg)
print(rf)
rf = train_rf(rf, X_train, y_train)

val_pred = rf.predict(X_val)
val_f1   = f1_score(y_val, val_pred, average="binary", pos_label=1)
print(f"Val F1: {val_f1:.4f}")

save_rf(rf, "../results/checkpoints/best_rf.pkl")

log_experiment({
    "exp_id": "rf_01_baseline",
    "model": "RandomForest",
    "best_val_f1": round(val_f1, 4),
    "notes": f"baseline n_est={cfg["random_forest"]["n_estimators"]} depth={cfg["random_forest"]["max_depth"]}",
}, csv_path=str(DRIVE_ROOT / "metrics_summary.csv"))

## Hyperparameter Experiments

In [ ]:
import copy
from sklearn.ensemble import RandomForestClassifier

def run_rf_experiment(exp_id, overrides, notes=""):
    exp_cfg = copy.deepcopy(cfg)
    exp_cfg["random_forest"].update(overrides)

    m = build_rf(exp_cfg)
    m = train_rf(m, X_train, y_train)
    val_pred = m.predict(X_val)
    best_f1  = round(f1_score(y_val, val_pred, average="binary", pos_label=1), 4)

    log_experiment({
        "exp_id": exp_id,
        "model": "RandomForest",
        "best_val_f1": best_f1,
        "notes": notes,
    }, csv_path=str(DRIVE_ROOT / "metrics_summary.csv"))
    print(f"[{exp_id}] val_F1={best_f1}")
    return m, best_f1

In [ ]:
_, f1_02 = run_rf_experiment('rf_02_trees50',   {'n_estimators': 50},          'n_estimators=50')

In [ ]:
_, f1_03 = run_rf_experiment('rf_03_trees200',  {'n_estimators': 200},         'n_estimators=200')

In [ ]:
_, f1_04 = run_rf_experiment('rf_04_depth10',   {'max_depth': 10},             'max_depth=10')

In [ ]:
_, f1_05 = run_rf_experiment('rf_05_depth20',   {'max_depth': 20},             'max_depth=20')

In [ ]:
_, f1_06 = run_rf_experiment('rf_06_minsplit5', {'min_samples_split': 5},      'min_samples_split=5')

## Feature Importance

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

features = cfg['features']
importances = rf.feature_importances_
fi_df = pd.DataFrame({'feature': features, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(fi_df['feature'], fi_df['importance'])
ax.set_xlabel('Importance')
ax.set_title('Random Forest — Feature Importance')
plt.tight_layout()
plt.savefig('../results/figures/rf_feature_importance.png', dpi=150)
plt.show()

## Final Evaluation on Test Set

In [ ]:
import pandas as pd
from src.models.rf_model import load_rf
from src.evaluate import evaluate_rf_model

# Print experiment table
results = pd.read_csv(str(DRIVE_ROOT / "metrics_summary.csv"))
rf_results = results[results['model'] == 'RandomForest']
avail = [c for c in ['exp_id', 'n_estimators', 'max_depth', 'best_val_f1', 'notes'] if c in rf_results.columns]
print(rf_results[avail])

# Evaluate best checkpoint on test set
best_rf = load_rf('../results/checkpoints/best_rf.pkl')
y_pred_rf, y_prob_rf = evaluate_rf_model(
    best_rf, X_test, y_test,
    figures_dir='../results/figures',
    csv_path=str(DRIVE_ROOT / "metrics_summary.csv"),
)

np.save('../results/rf_y_prob.npy', y_prob_rf)
np.save('../results/y_test_flat.npy', y_test)

In [ ]:
# Save best checkpoint to Drive so it persists across sessions
import shutil
shutil.copy("../results/checkpoints/best_rf.pkl", str(DRIVE_ROOT / "best_rf.pkl"))
np.save(str(DRIVE_ROOT / "rf_y_prob.npy"), y_prob_rf)
np.save(str(DRIVE_ROOT / "y_test_flat.npy"), y_test)
print(f"Checkpoint + probs saved to Drive: {DRIVE_ROOT}")